<!-- NB_DOC_INTRO_V1 -->
# Pandas — cheat-sheet exhaustive DataFrame

> 📚 **Doc thematique** : [docs/01_STRUCTURES.md](docs/01_STRUCTURES.md) (Structures Python / DataFrame / Preprocessing)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Cheat-sheet **pandas** complet : creation, selection, modification, manquants, types, groupby, joins (incl. merge_asof), reshape (pivot/melt/stack), strings/regex, datetime, window functions, multi-index, performance. Le notebook execute tous les patterns sur des datasets jouet reproductibles.

Pour Polars (alternative moderne Rust, 5-30x plus rapide sur gros volumes) : [Structures_Polars.ipynb](./Structures_Polars.ipynb).

## Plan

1. Setup + creation DataFrames
2. Selection (loc, iloc, query, where)
3. Filtrage + masks
4. Modification (assign, drop, rename, replace)
5. Manquants (NaN)
6. GroupBy + aggregations
7. Joins (merge, merge_asof, concat)
8. Reshape (pivot, melt, stack, unstack, crosstab)
9. Strings + regex (`.str` accessor)
10. Datetime (`.dt` accessor, resample, rolling)
11. Window functions
12. Multi-index
13. Performance (eval, query, chunksize)
14. Pieges et anti-patterns
15. References


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
SEED = 42
np.random.seed(SEED)
print(f"pandas : {pd.__version__}")

## 1. Creation

In [ ]:
# Depuis dict
df = pd.DataFrame({
    "id":       range(1, 11),
    "name":     [f"user_{i}" for i in range(1, 11)],
    "age":      np.random.randint(20, 60, 10),
    "salary":   np.random.uniform(30000, 100000, 10).round(2),
    "active":   np.random.choice([True, False], 10),
    "category": np.random.choice(list("ABC"), 10),
    "joined":   pd.date_range("2020-01-01", periods=10, freq="180D"),
})
df.head()

## 2. Selection — `loc`, `iloc`, `query`

In [ ]:
# Colonne(s)
print("Single col (Series) :", df["name"].head(2).tolist())
print("\nMulti col (DataFrame) :")
print(df[["name", "age"]].head(2))

# Lignes par label : .loc[index_label, col_label]
print("\ndf.loc[0:2, ['name', 'age']] :")
print(df.loc[0:2, ["name", "age"]])

# Lignes par position : .iloc[row_idx, col_idx]
print("\ndf.iloc[:3, [0, 2]] :")
print(df.iloc[:3, [0, 2]])

# query (lisible, evite chaining)
print("\ndf.query('age > 40 and active'):")
print(df.query("age > 40 and active"))

## 3. Filtrage + masks booleens

In [ ]:
# Mask simple
m = df["age"] > 40
print(f"Nb adultes 40+ : {m.sum()}")
print(df[m].head(3))

# Multi-conditions (& | ~ avec parentheses obligatoires)
print("\nMulti :")
print(df[(df["age"] > 30) & (df["category"].isin(["A", "B"]))].head(3))

# isin, between
print("\nbetween + isin :")
print(df[df["age"].between(30, 50) & df["category"].isin(["A"])].head(3))

## 4. Modification — `assign`, `drop`, `rename`

In [ ]:
# assign : ajoute / ecrase colonne (chainable)
df2 = df.assign(
    salary_eur=lambda x: x["salary"] * 0.92,
    age_group=lambda x: pd.cut(x["age"], bins=[0, 30, 50, 100], labels=["young", "mid", "senior"]),
)
print(df2[["name", "salary", "salary_eur", "age", "age_group"]].head())

# drop
df3 = df2.drop(columns=["salary_eur"])
df3 = df3.drop(index=[0, 1])
print(f"\nApres drop : shape={df3.shape}")

# rename
print("\nrename :")
print(df.rename(columns={"id": "user_id", "name": "username"}).columns.tolist())

## 5. Manquants — NaN

In [ ]:
# Injecter NaN pour demo
df_nan = df.copy()
df_nan.loc[[1, 3, 5], "salary"] = np.nan
df_nan.loc[[2, 4], "age"] = np.nan

print(f"NaN par col :\n{df_nan.isna().sum()}")

# Drop
print(f"\nApres dropna : {df_nan.dropna().shape}")
print(f"Apres dropna(subset=['salary']) : {df_nan.dropna(subset=['salary']).shape}")

# Fill
print("\nfillna (forward, backward, median) :")
print(df_nan["salary"].fillna(method="ffill").head())
print(df_nan["salary"].fillna(df_nan["salary"].median()).head())

# Interpolate (numerique)
print("\nInterpolate :")
print(df_nan["salary"].interpolate().head())

## 6. GroupBy + agg

In [ ]:
# Aggregation unique
print(df.groupby("category")["salary"].mean())

# Multi-agg + multi-col
result = df.groupby("category").agg(
    mean_salary=("salary", "mean"),
    max_age=("age", "max"),
    n_active=("active", "sum"),
    n_users=("id", "count"),
).round(2)
print("\n", result)

# Agg avec custom function
print("\nQuantiles :")
print(df.groupby("category")["salary"].quantile([0.25, 0.75]).round(0))

# Transform : meme shape que l'input (pas reduction)
df["salary_in_cat_pct"] = df.groupby("category")["salary"].transform(lambda x: x / x.sum())
print("\nTransform (% du total cat) :")
print(df[["name", "category", "salary", "salary_in_cat_pct"]].head())

## 7. Joins

In [ ]:
orders = pd.DataFrame({"order_id": [1, 2, 3, 4], "user_id": [1, 3, 2, 99], "amount": [100, 200, 150, 300]})
users = pd.DataFrame({"user_id": [1, 2, 3, 4, 5], "name": list("ABCDE")})

# Inner
print("INNER :")
print(orders.merge(users, on="user_id"))

# Left
print("\nLEFT :")
print(orders.merge(users, on="user_id", how="left"))

# Anti (left only) via indicator
m = orders.merge(users, on="user_id", how="left", indicator=True)
print("\nANTI (orders sans user) :")
print(m[m["_merge"] == "left_only"])

### 7.1 `merge_asof` — jointure 'la plus proche par le bas' (TS)

In [ ]:
prices = pd.DataFrame({
    "ts": pd.to_datetime(["2024-01-01 09:00", "2024-01-01 09:01", "2024-01-01 09:05"]),
    "price": [100.0, 101.5, 105.0],
}).sort_values("ts")

trades = pd.DataFrame({
    "ts": pd.to_datetime(["2024-01-01 09:00:30", "2024-01-01 09:04:00"]),
    "qty": [10, 5],
}).sort_values("ts")

# Pour chaque trade, recuperer le dernier prix connu
result = pd.merge_asof(trades, prices, on="ts", direction="backward")
print(result)

### 7.2 `concat`

In [ ]:
# Empilement (axis=0)
df_top = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
df_bot = pd.DataFrame({"a": [5, 6], "b": [7, 8]})
print("axis=0 :\n", pd.concat([df_top, df_bot], ignore_index=True))

# Cote-a-cote (axis=1)
print("\naxis=1 :\n", pd.concat([df_top, df_bot.rename(columns={"a": "c", "b": "d"})], axis=1))

## 8. Reshape — pivot, melt, stack

In [ ]:
long_df = pd.DataFrame({
    "date":     ["2024-01", "2024-01", "2024-02", "2024-02"],
    "product":  ["A", "B", "A", "B"],
    "sales":    [100, 200, 150, 250],
})

# pivot : long → wide
wide = long_df.pivot(index="date", columns="product", values="sales")
print(f"PIVOT :\n{wide}")

# pivot_table : pivot + aggregation (utile si duplications)
print(f"\nPIVOT_TABLE :\n{long_df.pivot_table(index='date', columns='product', values='sales', aggfunc='sum')}")

# melt : wide → long
print(f"\nMELT :\n{wide.reset_index().melt(id_vars='date', var_name='product', value_name='sales')}")

# crosstab : compter les occurrences
print(f"\nCROSSTAB :\n{pd.crosstab(df['category'], df['active'])}")

## 9. Strings + regex — `.str` accessor

In [ ]:
text = pd.DataFrame({"email": ["alice@gmail.com", "BOB@yahoo.fr", "Charlie@hotmail.COM", "invalid"]})
text = text.assign(
    email_low=lambda x: x["email"].str.lower(),
    domain=lambda x: x["email"].str.split("@").str[1],            # safe : NaN si pas de @
    is_gmail=lambda x: x["email"].str.contains("gmail", case=False, na=False),
    local_part=lambda x: x["email"].str.extract(r"^([^@]+)@", expand=False),
    len_chars=lambda x: x["email"].str.len(),
)
text

## 10. Datetime — `.dt` accessor, resample, rolling

In [ ]:
ts = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=100, freq="D"),
    "value": np.random.randn(100).cumsum() + 100,
})

# .dt accessors
ts = ts.assign(
    year=lambda x: x["date"].dt.year,
    month=lambda x: x["date"].dt.month,
    dow=lambda x: x["date"].dt.day_name(),
    week=lambda x: x["date"].dt.isocalendar().week,
)
print(ts.head())

# resample (downsampling : daily → weekly)
weekly = ts.set_index("date")["value"].resample("W").mean()
print(f"\nWeekly resample (5 premiers) :\n{weekly.head()}")

# rolling (fenetre glissante)
ts["roll_7"] = ts["value"].rolling(window=7).mean()
ts["roll_30"] = ts["value"].rolling(window=30, min_periods=1).mean()
print(f"\nRolling :\n{ts[['date', 'value', 'roll_7', 'roll_30']].tail()}")

## 11. Window functions (`shift`, `rank`, `cumsum`)

In [ ]:
s = df.copy()
s = s.assign(
    salary_rank=lambda x: x.groupby("category")["salary"].rank(ascending=False, method="dense"),
    salary_pct_in_cat=lambda x: x.groupby("category")["salary"].rank(pct=True),
    cumsum_in_cat=lambda x: x.groupby("category")["salary"].cumsum(),
    prev_age=lambda x: x.groupby("category")["age"].shift(1),
)
print(s[["name", "category", "salary", "salary_rank", "cumsum_in_cat", "prev_age"]].head(10))

## 12. Multi-index

In [ ]:
mi = df.set_index(["category", "id"])
print(f"Multi-index : {mi.index.names}")
print(mi.head())

# Selection
print("\nLoc multi-level :")
print(mi.loc["A"])
print("\nxs (cross-section sur 1 level) :")
print(mi.xs("A", level="category").head())

# swap levels
print(f"\nswaplevel : {mi.swaplevel().index.names}")

## 13. Performance — `eval`, `query`, `chunksize`, dtype optim

In [ ]:
# eval : evaluation vectorisee de strings (+ rapide sur gros DF)
big = pd.DataFrame(np.random.randn(100_000, 5), columns=list("ABCDE"))

import time
t0 = time.perf_counter()
r1 = big["A"] + big["B"] * big["C"] - big["D"]
t1 = time.perf_counter() - t0

t0 = time.perf_counter()
r2 = big.eval("A + B * C - D")
t2 = time.perf_counter() - t0

print(f"Vectorise pandas : {t1*1000:.2f} ms")
print(f"eval()           : {t2*1000:.2f} ms  ({t1/t2:.1f}x)")

# Dtype optim : int64 -> int8/16 si possible
print(f"\nMemoire avant : {df.memory_usage(deep=True).sum():,} bytes")
df_opt = df.copy()
df_opt["age"] = pd.to_numeric(df_opt["age"], downcast="integer")
df_opt["category"] = df_opt["category"].astype("category")
print(f"Memoire apres : {df_opt.memory_usage(deep=True).sum():,} bytes")

# Read large CSV par chunks (no demo file ici)
# for chunk in pd.read_csv('big.csv', chunksize=100_000):
#     process(chunk)

## 14. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| `for row in df.iterrows()` | Vectoriser ou `df.apply(func, axis=1)` ou Polars |
| `df[col] = ...` quand SettingWithCopyWarning | `df.loc[mask, col] = ...` ou `.assign` |
| `.append()` dans une boucle | `pd.concat([df, ...])` une fois ou liste de dicts → DataFrame |
| `df.fillna(0)` sans reflechir | Comprendre MCAR/MAR/MNAR, utiliser median/mean/KNN selon contexte |
| Operations chaine sur `.str` Series | Cache via `.str` ou Polars (plus rapide) |
| `merge` sans verifier `validate=` | `validate='one_to_one'` / `'many_to_one'` etc pour eviter explosions |
| `to_csv` pour stockage > 100 MB | Parquet (×10 vitesse, /5 taille) |
| Inplace=True partout | Plus difficile a chainer, ne marche pas avec lambda/pipe |
| Pas de `pd.set_option('mode.copy_on_write', True)` | Pandas 2+ : opt-in pour eviter SettingWithCopyWarning |


## References

### Documentation
- pandas docs : https://pandas.pydata.org/docs/
- 10 minutes to pandas : https://pandas.pydata.org/docs/user_guide/10min.html
- Modern Pandas (Tom Augspurger) : https://tomaugspurger.net/posts/modern-1-intro/
- Pandas cookbook : https://github.com/jvns/pandas-cookbook

### Voir aussi
- [Structure_Pyhton.ipynb](Structure_Pyhton.ipynb)
- [Structures_Polars.ipynb](Structures_Polars.ipynb)
- [Structures_Preprocessing.ipynb](Structures_Preprocessing.ipynb)
- [Structures_L_T_D_Cheat_Sheet.ipynb](Structures_L_T_D_Cheat_Sheet.ipynb)


<!-- ENRICH_2025_V1 -->

## 🔥 Mises a jour pandas 2024-2025

Pandas 2+ a apporte plusieurs evolutions majeures qui changent les bonnes pratiques :

### 1. **PyArrow backend** (~2-5× moins de memoire, faster string ops)
```python
df = pd.read_csv("file.csv", dtype_backend="pyarrow")
# Ou explicit :
df = pd.read_csv("file.csv", engine="pyarrow")

# Toutes les colonnes en types Arrow → strings, dates, nullable int natif
print(df.dtypes)   # string[pyarrow], int64[pyarrow], ...
```

**Quand l'utiliser** : datasets > 1M lignes, beaucoup de strings, interop avec Polars/DuckDB.

### 2. **Copy-on-Write (CoW)** — defaut en 2.2+

Resout enfin le `SettingWithCopyWarning` ambigu. Les vues deviennent **toujours** des copies lors d'une modification.

```python
pd.options.mode.copy_on_write = True   # opt-in en 2.0-2.1, default 2.2+
```

Plus de `df.loc[mask, "col"] = value` qui modifie ou pas selon les conditions. **Toujours predictible**.

### 3. **`.assign()` + `.pipe()` + `.query()` chainables** — pattern moderne
```python
result = (
    df
    .query("age > 18 and country == 'FR'")
    .assign(salary_eur=lambda x: x["salary_usd"] * 0.92)
    .pipe(lambda x: x.merge(other_df, on="id"))
    .groupby("category")
    .agg(mean_sal=("salary_eur", "mean"), n=("id", "count"))
)
```

### 4. **Alternatives serieuses 2025**
| Lib | Pour |
|---|---|
| **Polars** (Rust) | 5-30× plus rapide sur > 1M lignes, lazy + streaming |
| **DuckDB** | SQL embed, requete directe sur DataFrame/Parquet/CSV |
| **Modin** | API pandas, backend Ray/Dask, scale gratuit |
| **cuDF** (RAPIDS) | API pandas, backend GPU NVIDIA |

> 🎯 **Strategie 2025** : commencer en pandas (familier), passer a polars/DuckDB si > 1M lignes.
